# Fine-Tuning DeepSeek-R1-Distill-Qwen-1.5B for Financial Sentiment Analysis

**Authors**: Daniela Sameny, Alexandre Georges Lissoko — Aivancity 2025–2026

## What this notebook does

Fine-tunes the open-source **DeepSeek-R1-Distill-Qwen-1.5B** model on the **Financial PhraseBank** dataset using **LoRA (Low-Rank Adaptation)** to specialize it on financial sentiment classification.

## Pipeline

1. Load Financial PhraseBank (~4,845 expert-labeled financial phrases)
2. Load DeepSeek-R1-Distill-Qwen-1.5B model + tokenizer
3. Configure LoRA adapters (rank=8, alpha=16) — only ~0.1% of params trained
4. Train 3 epochs on a T4 GPU (~45-60 minutes)
5. Evaluate accuracy + F1 on a held-out test set
6. Compare zero-shot vs fine-tuned performance
7. Save the LoRA adapter (small file, ~10 MB)

## Why this setup

- **DeepSeek-R1-Distill-Qwen-1.5B**: official DeepSeek open-source model
- **Financial PhraseBank**: standard academic benchmark (Malo et al. 2014)
- **LoRA**: trains only adapter weights, not the full model
- **Reproducible**: seed = 42 throughout

## Requirements

- Google Colab with **GPU T4 enabled** (Edit → Notebook settings → GPU)

## 1. Setup and GPU check

In [2]:
# Verify GPU is available
import subprocess
try:
    output = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])
    print(" GPU detected:")
    print(output.decode())
except Exception:
    print(" NO GPU detected!")
    print("   Go to: Edit → Notebook settings → Hardware accelerator → T4 GPU")
    print("   Then: Save → Runtime → Restart runtime")
    raise SystemExit("GPU is required.")

 GPU detected:
name, memory.total [MiB]
Tesla T4, 15360 MiB



In [3]:
# Install dependencies
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 28.0 MB/s eta 0:00:00


In [4]:
# Imports & reproducibility
import os
import random
import numpy as np
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
OUTPUT_DIR = "deepseek-finbert-lora"

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Load Financial PhraseBank dataset

Financial PhraseBank (Malo et al. 2014) contains 4,845 English sentences from financial news, each labeled by 16 expert annotators as **positive**, **neutral**, or **negative**.

We use the `sentences_allagree` subset where all annotators agreed (highest quality).

In [5]:
from datasets import load_dataset
import pandas as pd

print("Downloading financial sentiment dataset...")
dataset = load_dataset("nickmuchi/financial-classification")

print("Splits available:", list(dataset.keys()))
print("Columns:", dataset["train"].column_names)
print("First row:", dataset["train"][0])

# Combine train + test, we will redo the split ourselves with our seed
df_train = dataset["train"].to_pandas()
df_test  = dataset["test"].to_pandas()
df = pd.concat([df_train, df_test], ignore_index=True)

# Find the right column names
text_col  = "sentence"  if "sentence" in df.columns else ("text"      if "text"      in df.columns else df.columns[0])
label_col = "label"     if "label"    in df.columns else ("sentiment" if "sentiment" in df.columns else df.columns[1])

df = df.rename(columns={text_col: "sentence", label_col: "label"})
df = df[["sentence", "label"]].dropna().reset_index(drop=True)

# Normalize label format
if df["label"].dtype == object:
    df["label_text"] = df["label"].astype(str).str.lower().str.strip()
    label_to_id = {"negative": 0, "neutral": 1, "positive": 2}
    df["label"] = df["label_text"].map(label_to_id)
else:
    label_map = {0: "negative", 1: "neutral", 2: "positive"}
    df["label_text"] = df["label"].map(label_map)

df = df.dropna(subset=["label", "label_text"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)

print(f"\nLoaded {len(df)} sentences")
print(df["label_text"].value_counts())
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Splits available: ['train', 'test']
Columns: ['text', 'labels']
First row: {'text': 'Finnish airline Finnair is starting the temporary layoffs of cabin crews in February 2010 .', 'labels': 0}

Loaded 5057 sentences
label_text
neutral     2944
positive    1425
negative     688
Name: count, dtype: int64


,sentence,label,label_text
0,Finnish airline Finnair is starting the tempor...,0,negative
1,The corresponding increase in the share capita...,1,neutral
2,In the third quarter of fiscal 2008 Efore swun...,0,negative
3,"ALEXANDRIA , Va. , Oct. 15 -- Aaron Moss of Ha...",1,neutral
4,Vaisala Oyj Stock exchange release 26.03.2010 ...,1,neutral


In [6]:
# Train/test split (stratified)
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.15, random_state=SEED, stratify=df["label"]
)
print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")

Train: 4298  |  Test: 759


## 3. Format examples as instruction-following prompts

We frame sentiment classification as instruction-following so the LLM can learn it via causal language modeling.

In [7]:
# Format each example as an instruction
SYSTEM = (
    "You are a financial expert analyst. "
    "Classify the sentiment of the following financial news sentence as "
    "exactly one of: positive, neutral, or negative. "
    "Respond with only that single word."
)

def format_example(sentence, label_text=None):
    """Build a prompt; if label_text is given, append it (training mode)."""
    user = f"Sentence: {sentence}\nSentiment:"
    if label_text is None:
        return f"<|im_start|>system\n{SYSTEM}<|im_end|>\n<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n"
    return (
        f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n{label_text}<|im_end|>"
    )

# Build train dataset
train_texts = [format_example(s, l) for s, l in zip(train_df["sentence"], train_df["label_text"])]
train_dataset = Dataset.from_dict({"text": train_texts})
print("Sample training example:")
print()
print(train_dataset[0]["text"])

Sample training example:

<|im_start|>system
You are a financial expert analyst. Classify the sentiment of the following financial news sentence as exactly one of: positive, neutral, or negative. Respond with only that single word.<|im_end|>
<|im_start|>user
Sentence: Growth was strongest in F-Secure 's operator ISPs , mobile operators and cable operators business .
Sentiment:<|im_end|>
<|im_start|>assistant
positive<|im_end|>


## 4. Load DeepSeek-R1-Distill-Qwen-1.5B with 4-bit quantization

We use **4-bit quantization** (QLoRA) so the model fits comfortably in T4 GPU memory.

In [8]:
# Configure 4-bit quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f" Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
print("✅ Model loaded")

 Loading deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✅ Model loaded


## 5. Configure LoRA adapters

LoRA injects small low-rank matrices into the attention layers. Only those are trained — the original 1.5B weights stay frozen.

- **rank r = 8** — adapter dimension
- **alpha = 16** — scaling factor
- **dropout = 0.05** — regularization
- **target_modules** = attention projections (q, k, v, o)

In [9]:
# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,779,267,072 || trainable%: 0.1225


## 6. Train (3 epochs, ~45-60 min on T4)

In [13]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    optim="paged_adamw_8bit",
    bf16=True,
    fp16=False,
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    seed=SEED,
)

try:
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        args=training_args,
        processing_class=tokenizer,
    )
    print("Trainer ready (new API)")
except TypeError:
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        args=training_args,
        tokenizer=tokenizer,
        dataset_text_field="text",
        max_seq_length=256,
        packing=False,
    )
    print("Trainer ready (legacy API)")

print("Starting training...")

Adding EOS to train dataset:   0%|          | 0/4298 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4298 [00:00<?, ? examples/s]

Trainer ready (new API)
Starting training...


In [14]:
!pip install -q --upgrade "transformers==4.46.3" "accelerate==1.1.1" "trl==0.12.2" "peft==0.13.2" "bitsandbytes==0.44.1" "datasets==3.1.0"

In [15]:
# Train!
trainer.train()
print("✅ Training complete")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,3.620013
40,1.858868
60,1.184039
80,1.160924
100,1.101731
120,1.044817
140,1.116034
160,1.139698
180,1.115882
200,1.123404


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


KeyboardInterrupt: 

In [16]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}/")
!ls -lh {OUTPUT_DIR}

Adapter saved to deepseek-finbert-lora/
total 16M
-rw-r--r-- 1 root root 1.1K May  5 21:06 adapter_config.json
-rw-r--r-- 1 root root 4.2M May  5 21:06 adapter_model.safetensors
-rw-r--r-- 1 root root 2.2K May  5 21:06 chat_template.jinja
drwxr-xr-x 2 root root 4.0K May  5 20:59 checkpoint-269
-rw-r--r-- 1 root root 1.6K May  5 21:06 README.md
-rw-r--r-- 1 root root  421 May  5 21:06 tokenizer_config.json
-rw-r--r-- 1 root root  11M May  5 21:06 tokenizer.json


In [18]:
model.config.use_cache = True
model.eval()

def predict_sentiment(sentence, max_new_tokens=10):
    prompt = format_example(sentence)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    decoded = decoded.lower().strip()
    if "positive" in decoded: return "positive"
    if "negative" in decoded: return "negative"
    if "neutral" in decoded: return "neutral"
    return "neutral"

samples = ["Apple reported record quarterly profits, beating analyst expectations.",
    "Tesla missed delivery targets for the third consecutive quarter.",
    "The Federal Reserve maintained interest rates as expected.",
]
for s in samples:
    print(f"  [{predict_sentiment(s):>8s}]  {s}")

  [positive]  Apple reported record quarterly profits, beating analyst expectations.
  [negative]  Tesla missed delivery targets for the third consecutive quarter.
  [ neutral]  The Federal Reserve maintained interest rates as expected.


## 7. Evaluate the fine-tuned model on the held-out test set

In [19]:
from tqdm.auto import tqdm

# Reduce test set size for speed (300 instead of full set)
test_subset = test_df.sample(n=min(300, len(test_df)), random_state=SEED).reset_index(drop=True)
print(f"Evaluating fine-tuned model on {len(test_subset)} test sentences...")

predictions = [predict_sentiment(s) for s in tqdm(test_subset["sentence"].tolist())]
ground_truth = test_subset["label_text"].tolist()

acc = accuracy_score(ground_truth, predictions)
f1 = f1_score(ground_truth, predictions, average="weighted")
print()
print(f"Fine-tuned DeepSeek results:")
print(f"   Accuracy : {acc:.4f}")
print(f"   F1 (weighted) : {f1:.4f}")
print()
print(classification_report(ground_truth, predictions, digits=4))

Evaluating fine-tuned model on 300 test sentences...


  0%|          | 0/300 [00:00<?, ?it/s]


Fine-tuned DeepSeek results:
   Accuracy : 0.8433
   F1 (weighted) : 0.8417

              precision    recall  f1-score   support

    negative     0.8039    0.8723    0.8367        47
     neutral     0.8634    0.8927    0.8778       177
    positive     0.8182    0.7105    0.7606        76

    accuracy                         0.8433       300
   macro avg     0.8285    0.8252    0.8250       300
weighted avg     0.8426    0.8433    0.8417       300



In [21]:
print("Reloading base model (zero-shot, no fine-tuning)...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.use_cache = True
base_model.eval()

def predict_baseline(sentence, max_new_tokens=10):
    prompt = format_example(sentence)
    inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        out = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    decoded = decoded.lower().strip()
    if "positive" in decoded: return "positive"
    if "negative" in decoded: return "negative"
    if "neutral" in decoded: return "neutral"
    return "neutral"

print(f"Evaluating zero-shot on {len(test_subset)} sentences...")
predictions_baseline = [predict_baseline(s) for s in tqdm(test_subset["sentence"].tolist())]

acc_baseline = accuracy_score(ground_truth, predictions_baseline)
f1_baseline = f1_score(ground_truth, predictions_baseline, average="weighted")

print()
print(f"Zero-shot DeepSeek (no fine-tuning):")
print(f"   Accuracy : {acc_baseline:.4f}")
print(f"   F1 (weighted) : {f1_baseline:.4f}")

Reloading base model (zero-shot, no fine-tuning)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Evaluating zero-shot on 300 sentences...


  0%|          | 0/300 [00:00<?, ?it/s]


Zero-shot DeepSeek (no fine-tuning):
   Accuracy : 0.5400
   F1 (weighted) : 0.4532


In [20]:
# Inference helper
model.config.use_cache = True
model.eval()

def predict_sentiment(sentence, max_new_tokens=10):
    prompt = format_example(sentence)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    decoded = decoded.lower().strip()
    if "positive" in decoded: return "positive"
    if "negative" in decoded: return "negative"
    if "neutral" in decoded: return "neutral"
    return "neutral"

# Quick smoke test
samples = [
    "Apple reported record quarterly profits, beating analyst expectations.",
    "Tesla missed delivery targets for the third consecutive quarter.",
    "The Federal Reserve maintained interest rates as expected.",
]
for s in samples:
    print(f"  [{predict_sentiment(s):>8s}]  {s}")

  [positive]  Apple reported record quarterly profits, beating analyst expectations.
  [negative]  Tesla missed delivery targets for the third consecutive quarter.
  [ neutral]  The Federal Reserve maintained interest rates as expected.


## 8. Compare with zero-shot baseline (no fine-tuning)

To prove fine-tuning actually helped, we compare against the **same model without our adapter**.

## 9. Demo: apply the fine-tuned model to financial news headlines

In [22]:
import pandas as pd
results = pd.DataFrame({
    "Zero-shot (base)":        [acc_baseline, f1_baseline],
    "Fine-tuned (LoRA, ours)": [acc, f1],
}, index=["Accuracy", "F1 (weighted)"]).T

print("=" * 65)
print("  FINAL COMPARISON - Financial PhraseBank test subset (n=300)")
print("=" * 65)
print(results.round(4).to_string())
print("=" * 65)
print()
print(f"Fine-tuning improved accuracy by {(acc - acc_baseline)*100:+.2f} points")
print(f"Fine-tuning improved F1 by       {(f1 - f1_baseline)*100:+.2f} points")
print()
relative_acc = (acc - acc_baseline) / acc_baseline * 100
relative_f1  = (f1  - f1_baseline)  / f1_baseline  * 100
print(f"Relative improvement: +{relative_acc:.1f}% accuracy, +{relative_f1:.1f}% F1")

results.to_csv("finetuning_results.csv")
print()
print("Saved finetuning_results.csv")

  FINAL COMPARISON - Financial PhraseBank test subset (n=300)
                         Accuracy  F1 (weighted)
Zero-shot (base)           0.5400         0.4532
Fine-tuned (LoRA, ours)    0.8433         0.8417

Fine-tuning improved accuracy by +30.33 points
Fine-tuning improved F1 by       +38.85 points

Relative improvement: +56.2% accuracy, +85.7% F1

Saved finetuning_results.csv


In [24]:
# Demo on real-world financial news headlines
demo_headlines = [
    "Apple reports record quarterly revenue, beating Wall Street expectations.",
    "Tesla shares plummet after disappointing delivery numbers.",
    "Microsoft announces a strategic partnership with OpenAI.",
    "Amazon misses Q3 earnings estimates, stock drops 8% after-hours.",
    "Google parent Alphabet authorizes new $70 billion stock buyback.",
    "The Federal Reserve raises interest rates by 25 basis points.",
    "Meta cuts 10,000 jobs in second wave of layoffs.",
    "NVIDIA stock hits all-time high on strong AI demand.",
]

print(" Fine-tuned DeepSeek predictions:")
print()
for h in demo_headlines:
    pred = predict_sentiment(h)
    emoji = {"positive": "", "negative": "", "neutral": ""}[pred]
    print(f"  {emoji} [{pred:>8s}]  {h}")

 Fine-tuned DeepSeek predictions:

   [positive]  Apple reports record quarterly revenue, beating Wall Street expectations.
   [negative]  Tesla shares plummet after disappointing delivery numbers.
   [ neutral]  Microsoft announces a strategic partnership with OpenAI.
   [negative]  Amazon misses Q3 earnings estimates, stock drops 8% after-hours.
   [ neutral]  Google parent Alphabet authorizes new $70 billion stock buyback.
   [ neutral]  The Federal Reserve raises interest rates by 25 basis points.
   [negative]  Meta cuts 10,000 jobs in second wave of layoffs.
   [positive]  NVIDIA stock hits all-time high on strong AI demand.
